# 02 — Huấn luyện EFormExcelMapper từ dataset đã tạo

Notebook này **chỉ** đọc đầu ra của `01_BuildDatasetComplete.ipynb` tại `tool/artifacts/dataset`. Không đọc mapping candidate thô và không tự coi gợi ý của AI là ground truth.

Đầu ra model: `tool/artifacts/model/EFormExcelMapper-v1`.

In [ ]:
from pathlib import Path
import json
import random

RUN_ON_COLAB = False
MODULE_ROOT_OVERRIDE = ""  # Chỉ điền khi notebook không nằm trong repository.

if RUN_ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    MODULE_ROOT = Path("/content/drive/MyDrive/eform_btp/Module/AI Import")
elif MODULE_ROOT_OVERRIDE:
    MODULE_ROOT = Path(MODULE_ROOT_OVERRIDE).expanduser().resolve()
else:
    start = Path.cwd().resolve()
    MODULE_ROOT = None
    for base in [start, *start.parents]:
        if (base / "tool" / "01_BuildDatasetComplete.ipynb").exists():
            MODULE_ROOT = base
            break
        nested = base / "Module" / "AI Import"
        if (nested / "tool" / "01_BuildDatasetComplete.ipynb").exists():
            MODULE_ROOT = nested
            break
    if MODULE_ROOT is None:
        raise FileNotFoundError("Không tìm thấy Module/AI Import. Hãy đặt MODULE_ROOT_OVERRIDE.")

TOOL_ROOT = MODULE_ROOT / "tool"
DATASET_ROOT = TOOL_ROOT / "artifacts" / "dataset"
TRAIN_PATH = DATASET_ROOT / "Train" / "train.jsonl"
VAL_PATH = DATASET_ROOT / "Validation" / "validation.jsonl"
TEST_PATH = DATASET_ROOT / "Test" / "test.jsonl"
GROUND_TRUTH_PATH = DATASET_ROOT / "Test" / "ground_truth.jsonl"
MODEL_OUTPUT = TOOL_ROOT / "artifacts" / "model" / "EFormExcelMapper-v1"
MODEL_OUTPUT.mkdir(parents=True, exist_ok=True)
print({"dataset": str(DATASET_ROOT), "model_output": str(MODEL_OUTPUT)})

In [ ]:
# Chạy một lần cho mỗi môi trường mới. Trên Colab nên bật GPU trước khi chạy.
%pip install -q -r "{TOOL_ROOT / 'runtime' / 'requirements-embedding.txt'}"

In [ ]:
def read_jsonl(path):
    if not path.is_file():
        raise FileNotFoundError(f"Thiếu {path}. Hãy chạy notebook 01 trước.")
    rows = []
    with path.open(encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, 1):
            if line.strip():
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f"JSONL lỗi tại {path}:{line_number}: {exc}") from exc
    return rows

def positive_text(row):
    positives = row.get("pos") or []
    return row.get("positive") or (positives[0] if positives else "")

train_rows = read_jsonl(TRAIN_PATH)
validation_rows = read_jsonl(VAL_PATH)
test_rows = read_jsonl(TEST_PATH)
ground_truth = read_jsonl(GROUND_TRUTH_PATH)

for split_name, rows, needs_answer in [
    ("train", train_rows, True),
    ("validation", validation_rows, True),
    ("test", test_rows, False),
]:
    ids = [row.get("sample_id") for row in rows]
    assert all(ids), f"{split_name}: thiếu sample_id"
    assert len(ids) == len(set(ids)), f"{split_name}: trùng sample_id"
    assert all(row.get("query") for row in rows), f"{split_name}: thiếu query"
    if needs_answer:
        assert all(positive_text(row) for row in rows), f"{split_name}: thiếu positive"

train_files = {row.get("source_file") for row in train_rows}
validation_files = {row.get("source_file") for row in validation_rows}
test_files = {row.get("source_file") for row in test_rows}
assert not (train_files & validation_files or train_files & test_files or validation_files & test_files), "Rò rỉ workbook giữa các split"
assert {row["sample_id"] for row in test_rows} == {row["sample_id"] for row in ground_truth}, "Test và ground truth lệch sample_id"
if len(train_rows) < 20:
    raise ValueError("Cần ít nhất 20 mẫu train đã được người duyệt xác nhận.")
print({"train": len(train_rows), "validation": len(validation_rows), "test": len(test_rows), "quality": "passed"})

In [ ]:
# Có thể giảm BATCH_SIZE nếu GPU thiếu VRAM.
SEED = 42
MODEL_NAME = "BAAI/bge-m3"
EPOCHS = 10
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.10
MAX_HARD_NEGATIVES = 4
random.seed(SEED)
print({"base_model": MODEL_NAME, "epochs": EPOCHS, "batch_size": BATCH_SIZE})

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)

def training_texts(row):
    positive = positive_text(row)
    negatives = [text for text in (row.get("neg") or []) if text and text != positive][:MAX_HARD_NEGATIVES]
    return [row["query"], positive, *negatives]

examples = [InputExample(texts=training_texts(row)) for row in train_rows]
loader = DataLoader(examples, shuffle=True, batch_size=BATCH_SIZE)
loss = losses.MultipleNegativesRankingLoss(model)
warmup_steps = max(1, int(len(loader) * EPOCHS * WARMUP_RATIO))
model.fit(
    train_objectives=[(loader, loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": LEARNING_RATE},
    output_path=str(MODEL_OUTPUT),
    show_progress_bar=True,
)
print({"device": device, "saved_model": str(MODEL_OUTPUT)})

In [ ]:
# Retrieval accuracy trên validation: positive đúng phải đứng đầu catalog ứng viên.
import numpy as np

catalog = sorted({positive_text(row) for row in [*train_rows, *validation_rows] if positive_text(row)})
if validation_rows and catalog:
    query_embeddings = model.encode([row["query"] for row in validation_rows], normalize_embeddings=True, show_progress_bar=True)
    target_embeddings = model.encode(catalog, normalize_embeddings=True, show_progress_bar=True)
    predicted = np.asarray(query_embeddings) @ np.asarray(target_embeddings).T
    correct = sum(catalog[int(scores.argmax())] == positive_text(row) for row, scores in zip(validation_rows, predicted))
    top1_accuracy = correct / len(validation_rows)
else:
    top1_accuracy = None

metrics = {
    "base_model": MODEL_NAME,
    "train_samples": len(train_rows),
    "validation_samples": len(validation_rows),
    "test_samples": len(test_rows),
    "validation_top1_accuracy": top1_accuracy,
}
(MODEL_OUTPUT / "training-metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(metrics, ensure_ascii=False, indent=2))

## Sau khi huấn luyện

1. Kiểm tra `training-metrics.json` và đánh giá riêng trên test + ground truth.
2. Chỉ phát hành model nếu kết quả tốt hơn baseline và không sai các biểu quan trọng.
3. Cấu hình service dùng thư mục model mới qua `AI_IMPORT_EMBEDDING_MODEL`.
4. Giữ LLM hierarchy mapping và validator JSON làm lớp kiểm soát; embedding model không thay thế kiểm tra cấp cha/con.